In [ ]:
import torch
import torch.nn as nn
from sklearn.datasets import make_moons
import matplotlib as mpl
import matplotlib.pyplot as plt
from tqdm import tqdm
import numpy as np
from utils import InputHook

In [ ]:
# Utility Functions

def closest_point_index(point, matrix):
    point = np.array(point)
    matrix = np.array(matrix)
    distances = np.linalg.norm(matrix - point, axis=1)
    return np.argmin(distances)

def clean_ax(ax):
    ax.set_xticks([])
    ax.set_xticklabels([])
    ax.set_yticks([])
    ax.set_yticklabels([])

def plot_heatmap(grid, fp, distances, separate_colorbar=False):
    distances=np.asarray(distances)
    fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(4, 4), dpi=150)

    sc = ax.scatter(
        grid[:, 0],
        grid[:, 1],
        c=distances,
        cmap="jet",
        s=4
    )
    ax.scatter(fp[0], fp[1], color=plt.cm.spring(0.25), s=20)

    clean_ax(ax)

    if separate_colorbar:
        fig_cb, ax_cb = plt.subplots(figsize=(0.3, 3), dpi=150)

        norm = mpl.colors.Normalize(
            vmin=distances.min(),
            vmax=distances.max()
        )
        cb = mpl.colorbar.ColorbarBase(
            ax_cb,
            cmap="jet",
            norm=norm,
            orientation="vertical",
        )

        ax_cb.set_ylabel("VQ Distance")
        plt.show()

    else:
        plt.show()
        return fig

In [ ]:
# Data Hyperparameters
NSAMPLES=256
NOISE=0.15

# Training Hyperparameters
EPOCHS=500
NHIDDEN=32
LR=0.01

# VQ Parameters
BETA=0.8

# Device
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# Reproducibility
SEED=0

np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

### Generating Data

In [ ]:
X,y=make_moons(n_samples=NSAMPLES,noise=NOISE,random_state=0)
X=X-X.mean(axis=0)


fig,ax=plt.subplots(nrows=1,ncols=1,figsize=(4,4),dpi=150)
ax.scatter(X[:,0],X[:,1],c=y,cmap='jet',alpha=0.75,edgecolor='k')
clean_ax(ax)
plt.show()

X=torch.tensor(X,dtype=torch.float32).to(DEVICE)
y=torch.tensor(y,dtype=torch.float32).to(DEVICE)

### Construct Model

In [ ]:
model = nn.Sequential(
    nn.Linear(2,NHIDDEN),
    nn.ReLU(),
    nn.Linear(NHIDDEN,NHIDDEN),
    nn.ReLU(),
    nn.Linear(NHIDDEN,1)
    )
model.to(DEVICE)

### Train Model

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(),lr=0.01)

pbar=tqdm(range(EPOCHS))
for epoch in pbar:
    optimizer.zero_grad()
    output=model(X).squeeze(1)
    loss=nn.BCEWithLogitsLoss()(output,y)
    loss.backward()
    optimizer.step()
    if epoch%10==0:
        pbar.set_description(f'Loss: {loss.item():.4f}')

### Create Sample Grid

In [ ]:
nx, ny = 120, 120
x = torch.linspace(-2, 2, nx)
y = torch.linspace(-2, 2, ny)

X_g, Y_g = torch.meshgrid(x, y, indexing="ij")

grid = torch.stack([X_g, Y_g], dim=-1).reshape(-1, 2).to(DEVICE)
grid_np=grid.detach().cpu().numpy()

### Fix Training Sample

In [ ]:
fixed_point_id=closest_point_index([-0.25,-0.1],X.detach().cpu().numpy())
fp_t=X[fixed_point_id].to(DEVICE)
fp_np=fp_t.detach().cpu().numpy()

### Collect Hard and Soft VQ Distances

In [ ]:
hook=InputHook(model, beta=1.0)
hardvq_distances=[]
with torch.no_grad():
    for p in grid:
        hook(torch.vstack([p,fp_t]))
        hard_vqk=hook.vq_kernel
        hardvq_distances.append(hard_vqk[1,0].item())
hook.remove()

hook=InputHook(model, beta=BETA)
softvq_distances=[]
with torch.no_grad():
    for p in grid:
        hook(torch.vstack([p,fp_t]))
        soft_vqk=hook.vq_kernel
        softvq_distances.append(soft_vqk[1,0].item())
hook.remove()

### Plot Heat Map of VQ Distances

In [ ]:
print("HardVQ Heatmap")
plot_heatmap(grid_np,fp_np,hardvq_distances)


print("SoftVQ Heatmap")
plot_heatmap(grid_np,fp_np,softvq_distances,separate_colorbar=True)